In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronce.matriz")
print("Filas leídas de bronce:", df.count())


Filas leídas de bronce: 460


In [0]:
# Espacios y saltos de línea en las orillas del texto
ORILLAS = r"^[ \t\u00A0\r\n]+|[ \t\u00A0\r\n]+$"
# Saltos de línea en medio del texto (con los espacios que tengan alrededor)
SALTOS = r"[ \t\u00A0]*[\r\n]+[ \t\u00A0\r\n]*"


# Devuelve la columna limpia: sin espacios en las orillas, saltos cambiados por comas
def limpiar_texto(columna):
    valor = F.col(columna)
    valor = F.regexp_replace(valor, ORILLAS, "")
    valor = F.regexp_replace(valor, SALTOS, ", ")
    return F.when(valor == "", F.lit(None)).otherwise(valor)


In [0]:
# Limpiar todas las columnas de texto (todas menos fecha_carga)
for columna in df.columns:
    if columna != "fecha_carga":
        df = df.withColumn(columna, limpiar_texto(columna))

# Eliminar la columna que está completamente vacía
df = df.drop("lider_de_la_fabrica")

print("Columnas:", len(df.columns))


Columnas: 33


In [0]:
# ¿Quedó algún salto de línea en Lenguaje de Desarrollo?
con_saltos = df.filter(F.col("lenguaje_de_desarrollo").contains("\n")).count()
print("Filas con saltos de línea:", con_saltos)

# Ver cómo quedaron las primeras filas
df.select("clave_o_id", "lenguaje_de_desarrollo", "fabrica_responsable").show(5, truncate=False)


Filas con saltos de línea: 0
+----------+----------------------------------------------------------------------+-------------------+
|clave_o_id|lenguaje_de_desarrollo                                                |fabrica_responsable|
+----------+----------------------------------------------------------------------+-------------------+
|1         |Informix 4GL 10, Visual Basic 6.0, Tuxedo 8                           |Fábrica SDMA 7 DS  |
|2         |Visual Studio 6.0                                                     |Fábrica SDMA 7 DS  |
|12        |Asp, Visual Basic 6.0, XML, .NET VS 2010 framework 4.0, Informix 11.11|Fábrica SDMA 7 DS  |
|17        |.NET                                                                  |Fábrica SDMA 7 DS  |
|23        |Informix 4GL, Visual Basic 6.0 Tuxedo 8                               |Fábrica SDMA 7 DS  |
+----------+----------------------------------------------------------------------+-------------------+
only showing top 5 rows


In [0]:
tabla_combos = spark.table("workspace.bronce.combos").drop("fecha_carga")
filas = tabla_combos.collect()

combos = {}
for columna in tabla_combos.columns:
    valores = []
    for fila in filas:
        valor = fila[columna]
        if valor is not None:
            valores.append(valor.strip())
    combos[columna] = valores

for columna, valores in combos.items():
    print(columna, "->", valores)


aplicacion_critica -> ['Crítica', 'No crítica', 'Por definir']
sla_s -> ['Sí', 'No']
criticidad_remedy -> ['Alta', 'Media', 'Baja']
aplicacion_en_la_nube -> ['Sí', 'No']
aplicacion_o_componente -> ['Aplicación', 'Componente', 'Por definir']
ventanilla_unica_gob_mx -> ['Sí', 'No', 'Por definir']
utiliza_el_firmado_electronico_de_documentos_y_o_tramites -> ['Sí', 'No', 'Por definir']
mesa_de_servicio -> ['Sí', 'No', 'Por definir']


In [0]:
# Versión Python: simplifica un texto del catálogo (minúsculas, sin acentos)
def simplificar_texto(texto):
    texto = texto.lower()
    texto = texto.replace("á", "a").replace("é", "e").replace("í", "i")
    texto = texto.replace("ó", "o").replace("ú", "u")
    return texto


# Versión Spark: simplifica una columna completa (minúsculas, sin acentos)
def simplificar_columna(columna):
    return F.translate(F.lower(F.col(columna)), "áéíóú", "aeiou")


# Cambia cada valor de la columna por su escritura oficial del catálogo
def normalizar_con_catalogo(columna, valores_permitidos):
    resultado = F.col(columna)
    for valor in valores_permitidos:
        resultado = F.when(
            simplificar_columna(columna) == simplificar_texto(valor), F.lit(valor)
        ).otherwise(resultado)
    return resultado


In [0]:
si_no = combos["mesa_de_servicio"]  # ["Sí", "No", "Por definir"]

df = df.withColumn("aplicacion_critica", normalizar_con_catalogo("aplicacion_critica", combos["aplicacion_critica"]))
df = df.withColumn("criticidad_remedy", normalizar_con_catalogo("criticidad_remedy", combos["criticidad_remedy"] + ["Por definir"]))
df = df.withColumn("aplicacion_o_componente", normalizar_con_catalogo("aplicacion_o_componente", combos["aplicacion_o_componente"]))
df = df.withColumn("aplicacion_en_la_nube", normalizar_con_catalogo("aplicacion_en_la_nube", si_no))
df = df.withColumn("mesa_de_servicio", normalizar_con_catalogo("mesa_de_servicio", si_no))
df = df.withColumn("pistas_auditoria", normalizar_con_catalogo("pistas_auditoria", si_no))


In [0]:
columnas_revisadas = [
    "aplicacion_critica",
    "criticidad_remedy",
    "aplicacion_o_componente",
    "aplicacion_en_la_nube",
    "mesa_de_servicio",
    "pistas_auditoria",
]

for columna in columnas_revisadas:
    df.groupBy(columna).count().orderBy(columna).show()


+------------------+-----+
|aplicacion_critica|count|
+------------------+-----+
|              NULL|    1|
|           Crítica|  161|
|        No crítica|  297|
|       Por definir|    1|
+------------------+-----+

+-----------------+-----+
|criticidad_remedy|count|
+-----------------+-----+
|             Alta|  167|
|             Baja|   47|
|            Media|  244|
|      Por definir|    2|
+-----------------+-----+

+-----------------------+-----+
|aplicacion_o_componente|count|
+-----------------------+-----+
|                   NULL|    1|
|             Aplicación|  367|
|             Componente|   82|
|            Por definir|   10|
+-----------------------+-----+

+---------------------+-----+
|aplicacion_en_la_nube|count|
+---------------------+-----+
|                 NULL|    1|
|                   No|  393|
|          Por definir|   12|
|                   Sí|   54|
+---------------------+-----+

+----------------+-----+
|mesa_de_servicio|count|
+----------------+-----+
|

In [0]:
df = df.withColumn("severidad_remedy", F.expr("try_cast(severidad_remedy AS INT)"))


In [0]:
# Reemplaza valores de una columna según un diccionario {valor_viejo: valor_nuevo}
def reemplazar_valores(columna, reemplazos):
    resultado = F.col(columna)
    for viejo, nuevo in reemplazos.items():
        resultado = F.when(F.col(columna) == viejo, F.lit(nuevo)).otherwise(resultado)
    return resultado


# Unificar las distintas escrituras de "3 capas"
df = df.withColumn("tipo_de_arquitectura", reemplazar_valores("tipo_de_arquitectura", {"3 Capas": "3 capas", "Tres capas": "3 capas"}))

# Pendiente de calidad: "No Aplica" / "No aplica" en Tipo Movilidad
df = df.withColumn("tipo_movilidad", normalizar_con_catalogo("tipo_movilidad", ["No Aplica", "Por definir"]))


In [0]:
total = df.count()
claves_distintas = df.select("clave_o_id").distinct().count()

print("Filas:", total)
print("Claves distintas:", claves_distintas)

if total != claves_distintas:
    raise Exception("Hay claves repetidas en clave_o_id. No se guarda la tabla plata.")

print("Validación correcta: cada aplicación tiene una clave única")


Filas: 460
Claves distintas: 460
Validación correcta: cada aplicación tiene una clave única


In [0]:
(
    df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.plata.aplicaciones")
)

print("Tabla plata.aplicaciones guardada")


Tabla plata.aplicaciones guardada


In [0]:
plata = spark.table("workspace.plata.aplicaciones")

plata.groupBy("severidad_remedy").count().orderBy("severidad_remedy").show()
plata.groupBy("tipo_movilidad").count().orderBy("tipo_movilidad").show()
print("Filas con '3 capas':", plata.filter(F.col("tipo_de_arquitectura") == "3 capas").count())


+----------------+-----+
|severidad_remedy|count|
+----------------+-----+
|            NULL|   10|
|               1|   26|
|               2|  112|
|               3|  262|
|               4|   50|
+----------------+-----+

+--------------------+-----+
|      tipo_movilidad|count|
+--------------------+-----+
|                NULL|    2|
|        Apps Móviles|    5|
|             Híbrida|   14|
|    Móvil Responsiva|    1|
|Móvil Responsiva ...|    1|
|             Ninguno|    7|
|           No Aplica|  418|
|         Por definir|    7|
|                 Web|    5|
+--------------------+-----+

Filas con '3 capas': 7
